In [3]:
import re
import faiss
import numpy as np
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer
from openai import OpenAI

In [4]:
EMBED_MODEL = "all-MiniLM-L6-v2"
TOP_K       = 5
CHUNK_SIZE  = 500
OVERLAP     = 100

# Paste your key here
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", "your-api-key-here"))
embedder = SentenceTransformer(EMBED_MODEL)  

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\twink\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\twink\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [32]:
import pdfplumber
import re

PDF_PATH = r"C:\Users\twink\OneDrive\Desktop\test_pdf.pdf"

raw_text = ""
with pdfplumber.open(PDF_PATH) as pdf:
    print(f"Total pages: {len(pdf.pages)}")
    for page in pdf.pages:
        text = page.extract_text(x_tolerance=2, y_tolerance=2)
        if text:
            raw_text += text + "\n"

# Fix squished words — add space before capitals in the middle of words
raw_text = re.sub(r'([a-z])([A-Z])', r'\1 \2', raw_text)
# Fix spacing
raw_text = re.sub(r'\s+', ' ', raw_text)
raw_text = "\n".join(line.strip() for line in raw_text.splitlines() if line.strip())

print(f"Extracted {len(raw_text):,} characters")
print("\nFirst 500 chars:")
print(raw_text[:500])

# Chunk
chunks, start = [], 0
while start < len(raw_text):
    chunks.append(raw_text[start : start + CHUNK_SIZE])
    start += CHUNK_SIZE - OVERLAP
chunks = [c for c in chunks if len(c.strip()) > 50]
print(f"Created {len(chunks)} chunks")

Total pages: 152
Extracted 571,972 characters

First 500 chars:
B H ERKSHIRE ATHAWAY INC. 2023 ANNUAL REPORT BERKSHIRE HATHAWAY INC. 2023 ANNUAL REPORT TABLE OF CONTENTS Charlie Munger – The Architect of Berkshire Hathaway . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 2 Chairman’s Letter* . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 3-16 Berkshire’s Performance vs. the S&P 500 . . . . . . . . . . . . . . . . . . . . . . .
Created 1430 chunks


In [33]:
print("Embedding chunks... (takes ~30 seconds)")
vecs  = embedder.encode(chunks, show_progress_bar=True).astype("float32")
index = faiss.IndexFlatL2(vecs.shape[1])
index.add(vecs)
print(f"Index ready — {index.ntotal} vectors")

Embedding chunks... (takes ~30 seconds)


Batches:   0%|          | 0/45 [00:00<?, ?it/s]

Index ready — 1430 vectors


In [34]:
query = "What are Berkshire Hathaway's main business segments"   # ← change this each time

# Retrieve top-K relevant chunks
q_vec      = embedder.encode([query]).astype("float32")
_, indices = index.search(q_vec, TOP_K)
context    = "\n\n---\n\n".join(chunks[i] for i in indices[0])

# Generate answer
prompt = f"""You are a financial analyst. Use ONLY the context below to answer.
If the answer isn't there, say so.

CONTEXT:
{context}

QUESTION: {query}
ANSWER:"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.2,
)

print("QUESTION:", query)
print("\nANSWER:")
print(response.choices[0].message.content)

QUESTION: What are Berkshire Hathaway's main business segments

ANSWER:
Berkshire Hathaway's main business segments include insurance, freight rail transportation, utility and energy generation and distribution, manufacturing, service, and retailing.


In [36]:
query = "What risks does Berkshire Hathaway mention?"

# Retrieve top-K relevant chunks
q_vec      = embedder.encode([query]).astype("float32")
_, indices = index.search(q_vec, TOP_K)
context    = "\n\n---\n\n".join(chunks[i] for i in indices[0])

# Generate answer
prompt = f"""You are a financial analyst. Use ONLY the context below to answer.
If the answer isn't there, say so.

CONTEXT:
{context}

QUESTION: {query}
ANSWER:"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.2,
)

print("QUESTION:", query)
print("\nANSWER:")
print(response.choices[0].message.content)

QUESTION: What risks does Berkshire Hathaway mention?

ANSWER:
Berkshire Hathaway mentions risks and uncertainties in its business operations, including potential legal actions that may assert claims or seek to impose fines and penalties. Additionally, it acknowledges the possibility of financial disasters and negative surprises, particularly in its insurance business, where the basic product is risk assumption. However, the specific risks are not detailed beyond these general statements.


In [37]:
query = "What does Warren Buffett say about the economy?"

# Retrieve top-K relevant chunks
q_vec      = embedder.encode([query]).astype("float32")
_, indices = index.search(q_vec, TOP_K)
context    = "\n\n---\n\n".join(chunks[i] for i in indices[0])

# Generate answer
prompt = f"""You are a financial analyst. Use ONLY the context below to answer.
If the answer isn't there, say so.

CONTEXT:
{context}

QUESTION: {query}
ANSWER:"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.2,
)

print("QUESTION:", query)
print("\nANSWER:")
print(response.choices[0].message.content)

QUESTION: What does Warren Buffett say about the economy?

ANSWER:
Warren Buffett suggests that markets and the economy can cause stocks and bonds of fundamentally good businesses to be mispriced. He notes that markets can unpredictably seize up or vanish, referencing historical events like those in 1914 and 2001, and emphasizes that American investors are not necessarily more stable than in the past, citing the events of September 2008. He also mentions that within capitalism, some businesses will flourish for a long time while others will fail, and predicting which will be winners or losers is challenging.


In [38]:
query = "What is Berkshire's investment strategy?"

# Retrieve top-K relevant chunks
q_vec      = embedder.encode([query]).astype("float32")
_, indices = index.search(q_vec, TOP_K)
context    = "\n\n---\n\n".join(chunks[i] for i in indices[0])

# Generate answer
prompt = f"""You are a financial analyst. Use ONLY the context below to answer.
If the answer isn't there, say so.

CONTEXT:
{context}

QUESTION: {query}
ANSWER:"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.2,
)

print("QUESTION:", query)
print("\nANSWER:")
print(response.choices[0].message.content)

QUESTION: What is Berkshire's investment strategy?

ANSWER:
Berkshire's investment strategy focuses on owning either all or a portion of businesses that enjoy good economics that are fundamental and enduring. The company seeks to invest in a limited number of companies that can significantly impact its performance, prioritizing those that are attractively priced and capable of long-term value creation. Berkshire's management emphasizes a decentralized approach to managing its diverse business activities and is responsible for significant capital allocation decisions.
